# Four-Way Visual Analysis (OOD — Scenes 0011–0015)
Comparing attention heatmaps across three model variants on **out-of-distribution** scenes:
1. **Source** — query image with marked point
2. **Vanilla** — original `indoor-lite-LA.ckpt`, no constraint
3. **Vanilla + Epipolar** — same weights, soft epipolar mask at inference (τ=10)
4. **Fine-Tuned** — `epipolar-run=50000.ckpt` trained with epipolar supervision

In [ ]:
%matplotlib inline
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

from config.defaultmf import get_cfg_defaults
from model.lightning_loftr import PL_LoFTR
from gt_epipolar import compute_fundamental_matrix

# --- EPIPOLAR MASK ---
def create_epipolar_mask_flat(F, H_feat, W_feat, query_pt, H_img=480, W_img=640, tau=10.0):
    p = np.array([query_pt[0], query_pt[1], 1.0])
    l_prime = F @ p
    a, b, c = l_prime
    y_feat, x_feat = np.mgrid[0:H_feat, 0:W_feat]
    x_img = (x_feat / W_feat) * W_img
    y_img = (y_feat / H_feat) * H_img
    distances = np.abs(a * x_img + b * y_img + c) / np.sqrt(a**2 + b**2)
    mask = np.exp(-distances / tau)
    return torch.from_numpy(mask.flatten()).float()

# --- CONFIG ---
config = get_cfg_defaults()
config.MATCHFORMER.BACKBONE_TYPE = 'litela'
config.MATCHFORMER.SCENS = 'indoor'
config.MATCHFORMER.RESOLUTION = (8, 4)
config.MATCHFORMER.COARSE.D_MODEL = 192
config.MATCHFORMER.COARSE.D_FFN = 192

# Load three model instances
model_vanilla   = PL_LoFTR(config, pretrained_ckpt='model/weights/indoor-lite-LA.ckpt').eval()
model_const     = PL_LoFTR(config, pretrained_ckpt='model/weights/indoor-lite-LA.ckpt').eval()
model_finetuned = PL_LoFTR(config, pretrained_ckpt='model/weights/epipolar-run=50000.ckpt').eval()

HW_layer_map = {'stage4_cross': (15, 20)}

# --- ATTENTION HOOK ---
def get_cross_attn_hook(name, layer, ux, uy, store_dict, F_matrix=None):
    def hook(model, input, output):
        x = input[0]
        B, N, C = x.shape
        MiniB = B // 2
        query = layer.q(x).reshape(B, N, layer.num_heads, C // layer.num_heads).permute(0, 1, 2, 3)
        kv = layer.kv(x).reshape(B, -1, 2, layer.num_heads, C // layer.num_heads).permute(2, 0, 1, 3, 4)
        if layer.cross:
            k1, k2 = kv[0].split(MiniB)
            key = torch.cat([k2, k1], dim=0)
        else:
            return
        Q = layer.feature_map(query).permute(0, 2, 1, 3)
        K_mat = layer.feature_map(key).permute(0, 2, 1, 3)
        H_feat, W_feat = HW_layer_map[name]
        feat_x = int((ux / 640.0) * W_feat)
        feat_y = int((uy / 480.0) * H_feat)
        query_idx = feat_y * W_feat + feat_x
        Q_single = Q[0, :, query_idx:query_idx+1, :]
        K_targets = K_mat[0, :, :, :]
        attn = torch.matmul(Q_single, K_targets.transpose(-2, -1))
        if F_matrix is not None:
            epipolar_mask = create_epipolar_mask_flat(F_matrix, H_feat, W_feat, (ux, uy))
            epipolar_mask = epipolar_mask.view(1, 1, -1).to(attn.device)
            attn = attn * epipolar_mask
        Z = 1 / (attn.sum(dim=-1, keepdim=True) + layer.eps)
        attn_final = attn * Z
        store_dict[name] = attn_final.mean(dim=0).squeeze(0).detach().cpu()
    return hook

# --- HELPERS ---
def get_image(path, resize=(640, 480)):
    img_bgr = cv2.imread(path)
    img_rgb = cv2.resize(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), resize)
    img_gray = cv2.resize(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY), resize)
    img_tensor = torch.from_numpy(img_gray).float() / 255.0
    img_tensor = img_tensor.unsqueeze(0).unsqueeze(0)
    return img_rgb, img_tensor

def compute_ear(attn_prob, a, b, c, k_pixels):
    H, W = attn_prob.shape
    y, x = np.mgrid[0:H, 0:W]
    distances = np.abs(a * x + b * y + c) / np.sqrt(a**2 + b**2)
    mask = distances <= k_pixels
    return np.sum(attn_prob[mask])

def draw_epipolar_line(ax, a, b, c, w=640, h=480):
    x0, x1 = 0, w
    if abs(b) > 1e-8:
        y0 = int(-(a * x0 + c) / b)
        y1 = int(-(a * x1 + c) / b)
        ax.plot([x0, x1], [y0, y1], 'w--', linewidth=2, label='Epipolar Line')

def get_gt_match(pt1, depth1_path, pose1, pose2, K_intr):
    depth1 = cv2.imread(depth1_path, cv2.IMREAD_ANYDEPTH)
    if depth1 is None:
        return None
    if depth1.shape != (480, 640):
        depth1 = cv2.resize(depth1, (640, 480), interpolation=cv2.INTER_NEAREST)
    z = depth1[int(pt1[1]), int(pt1[0])] / 1000.0
    if z <= 0.1 or z > 10.0:
        return None
    cx, cy = K_intr[0, 2], K_intr[1, 2]
    fx, fy = K_intr[0, 0], K_intr[1, 1]
    x_c1 = (pt1[0] - cx) * z / fx
    y_c1 = (pt1[1] - cy) * z / fy
    p_c1 = np.array([x_c1, y_c1, z, 1.0])
    T_12 = np.linalg.inv(pose2) @ pose1
    p_c2 = T_12 @ p_c1
    if p_c2[2] <= 0:
        return None
    u2 = (p_c2[0] * fx / p_c2[2]) + cx
    v2 = (p_c2[1] * fy / p_c2[2]) + cy
    if 0 <= u2 < 640 and 0 <= v2 < 480:
        return np.array([u2, v2])
    return None

def heatmap_peak(heat_raw, H_feat, W_feat):
    heat_rez = cv2.resize(heat_raw.reshape(H_feat, W_feat), (640, 480))
    heat_norm = (heat_rez - heat_rez.min()) / (heat_rez.max() - heat_rez.min() + 1e-8)
    max_idx = np.unravel_index(np.argmax(heat_rez), heat_rez.shape)
    pred = np.array([max_idx[1], max_idx[0]])
    return heat_norm, heat_rez, pred

def overlay_pred(ax, pred, gt_match, a, b, c_coef, label_prefix):
    if gt_match is not None:
        dist = np.linalg.norm(pred - gt_match)
        color = 'g*' if dist < 10 else 'rx'
        ax.plot(pred[0], pred[1], color, markersize=12, label=f'{label_prefix}Pred (GT Err: {dist:.1f}px)')
    else:
        dist = np.abs(a * pred[0] + b * pred[1] + c_coef) / np.sqrt(a**2 + b**2)
        color = 'g*' if dist < 5.0 else 'rx'
        ax.plot(pred[0], pred[1], color, markersize=12, label=f'{label_prefix}Pred (Epi Err: {dist:.1f}px)')

print('Setup complete. Three models loaded.')

## Define plot_quad and scene data directories

In [ ]:
# Scene directories — scene0014 has flat structure, rest have exported/
SCENES = {
    'scene0011_00': '../data/scans/scene0011_00/exported',
    'scene0012_00': '../data/scans/scene0012_00/exported',
    'scene0013_00': '../data/scans/scene0013_00/exported',
    'scene0014_00': '../data/scans/scene0014_00',
    'scene0015_00': '../data/scans/scene0015_00/exported',
}

def load_scene(data_dir):
    """Load images, intrinsics for a scene."""
    all_imgs = sorted(glob.glob(f'{data_dir}/color/*.jpg'),
                      key=lambda x: int(os.path.basename(x).split('.')[0]))
    K_intr = np.loadtxt(f'{data_dir}/intrinsic/intrinsic_depth.txt')[:3, :3]
    return all_imgs, K_intr

def plot_quad(all_imgs, K_intr, data_dir, scene_name, img1_idx, img2_idx, query_points):
    img1_path = all_imgs[img1_idx]
    img2_path = all_imgs[img2_idx]
    img1_rgb, img1_tensor = get_image(img1_path)
    img2_rgb, img2_tensor = get_image(img2_path)

    img1_idx_num = os.path.basename(img1_path).split('.')[0]
    img2_idx_num = os.path.basename(img2_path).split('.')[0]

    try:
        T1 = np.loadtxt(os.path.join(data_dir, 'pose', f'{img1_idx_num}.txt'))
        T2 = np.loadtxt(os.path.join(data_dir, 'pose', f'{img2_idx_num}.txt'))
    except FileNotFoundError:
        print(f'Poses not found for pair ({img1_idx_num}, {img2_idx_num}), skipping.')
        return False

    if not np.isfinite(T1).all() or not np.isfinite(T2).all():
        print('Non-finite pose, skipping.')
        return False

    F = compute_fundamental_matrix(T1, T2, K_intr, K_intr)
    depth1_path = os.path.join(data_dir, 'depth', f'{img1_idx_num}.png')

    H_feat, W_feat = HW_layer_map['stage4_cross']

    for pt_name, (ux, uy) in query_points.items():
        attn_v = {}
        attn_c = {}
        attn_ft = {}

        # Register hooks on all three models
        h_v = model_vanilla.matcher.backbone.AttentionBlock4.block[2].attn.register_forward_hook(
            get_cross_attn_hook('stage4_cross',
                                model_vanilla.matcher.backbone.AttentionBlock4.block[2].attn,
                                ux, uy, attn_v, F_matrix=None))
        h_c = model_const.matcher.backbone.AttentionBlock4.block[2].attn.register_forward_hook(
            get_cross_attn_hook('stage4_cross',
                                model_const.matcher.backbone.AttentionBlock4.block[2].attn,
                                ux, uy, attn_c, F_matrix=F))
        h_ft = model_finetuned.matcher.backbone.AttentionBlock4.block[2].attn.register_forward_hook(
            get_cross_attn_hook('stage4_cross',
                                model_finetuned.matcher.backbone.AttentionBlock4.block[2].attn,
                                ux, uy, attn_ft, F_matrix=None))

        input_v  = {'image0': img1_tensor, 'image1': img2_tensor}
        input_c  = {'image0': img1_tensor, 'image1': img2_tensor}
        input_ft = {'image0': img1_tensor, 'image1': img2_tensor}

        with torch.no_grad():
            model_vanilla.matcher(input_v)
            model_const.matcher.coarse_matching.epipolar_F = F
            model_const.matcher(input_c)
            model_finetuned.matcher.coarse_matching.epipolar_F = None
            model_finetuned.matcher(input_ft)

        h_v.remove()
        h_c.remove()
        h_ft.remove()

        # Epipolar line coefficients
        p = np.array([ux, uy, 1.0])
        l_prime = F @ p
        a, b, c_coef = l_prime

        # Heatmaps & peaks
        heat_v_norm,  heat_v_rez,  pred_v  = heatmap_peak(attn_v['stage4_cross'].numpy(),  H_feat, W_feat)
        heat_c_norm,  heat_c_rez,  pred_c  = heatmap_peak(attn_c['stage4_cross'].numpy(),  H_feat, W_feat)
        heat_ft_norm, heat_ft_rez, pred_ft = heatmap_peak(attn_ft['stage4_cross'].numpy(), H_feat, W_feat)

        ear_v  = compute_ear(heat_v_rez  / (np.sum(heat_v_rez)  + 1e-8), a, b, c_coef, 10)
        ear_c  = compute_ear(heat_c_rez  / (np.sum(heat_c_rez)  + 1e-8), a, b, c_coef, 10)
        ear_ft = compute_ear(heat_ft_rez / (np.sum(heat_ft_rez) + 1e-8), a, b, c_coef, 10)

        gt_match = get_gt_match((ux, uy), depth1_path, T1, T2, K_intr)

        # --- PLOT ---
        fig, axes = plt.subplots(1, 4, figsize=(26, 6))
        fig.suptitle(f'{scene_name}  |  Pair ({img1_idx_num} \u2192 {img2_idx_num})  |  Query: {pt_name}  ({ux}, {uy})',
                     fontsize=13, fontweight='bold')

        # Panel 1 — Source
        axes[0].imshow(img1_rgb)
        axes[0].plot(ux, uy, 'r*', markersize=16)
        axes[0].set_title('Source Image', fontsize=11)
        axes[0].axis('off')

        # Panel 2 — Vanilla
        axes[1].imshow(img2_rgb)
        axes[1].imshow(heat_v_norm, cmap='jet', alpha=0.5)
        draw_epipolar_line(axes[1], a, b, c_coef)
        if gt_match is not None:
            axes[1].plot(gt_match[0], gt_match[1], 'yo', markersize=12, label='GT')
        overlay_pred(axes[1], pred_v, gt_match, a, b, c_coef, '')
        axes[1].set_title(f'Vanilla\nEAR: {ear_v:.4f}', fontsize=11)
        axes[1].legend(loc='lower right', fontsize=7)
        axes[1].set_xlim([0, 640]); axes[1].set_ylim([480, 0])
        axes[1].axis('off')

        # Panel 3 — Vanilla + Epipolar
        axes[2].imshow(img2_rgb)
        axes[2].imshow(heat_c_norm, cmap='jet', alpha=0.5)
        draw_epipolar_line(axes[2], a, b, c_coef)
        if gt_match is not None:
            axes[2].plot(gt_match[0], gt_match[1], 'yo', markersize=12, label='GT')
        overlay_pred(axes[2], pred_c, gt_match, a, b, c_coef, '')
        axes[2].set_title(f'Vanilla + Epipolar (\u03c4=10)\nEAR: {ear_c:.4f}', fontsize=11)
        axes[2].legend(loc='lower right', fontsize=7)
        axes[2].set_xlim([0, 640]); axes[2].set_ylim([480, 0])
        axes[2].axis('off')

        # Panel 4 — Fine-Tuned (epipolar-run=50000)
        axes[3].imshow(img2_rgb)
        axes[3].imshow(heat_ft_norm, cmap='jet', alpha=0.5)
        draw_epipolar_line(axes[3], a, b, c_coef)
        if gt_match is not None:
            axes[3].plot(gt_match[0], gt_match[1], 'yo', markersize=12, label='GT')
        overlay_pred(axes[3], pred_ft, gt_match, a, b, c_coef, '')
        axes[3].set_title(f'Fine-Tuned (run=50000)\nEAR: {ear_ft:.4f}', fontsize=11)
        axes[3].legend(loc='lower right', fontsize=7)
        axes[3].set_xlim([0, 640]); axes[3].set_ylim([480, 0])
        axes[3].axis('off')

        plt.tight_layout()
        plt.show()

    return True

print('plot_quad() defined.')

## Run: 100 random query points across scenes 0011–0015 (20 per scene)

In [ ]:
np.random.seed(42)
PAIRS_PER_SCENE = 20
FRAME_GAP = 20

total_plotted = 0

for scene_name, data_dir in SCENES.items():
    print(f'\n{"="*60}')
    print(f'Scene: {scene_name}')
    print(f'{"="*60}')

    all_imgs, K_intr = load_scene(data_dir)
    n_imgs = len(all_imgs)
    print(f'  {n_imgs} frames available')

    valid = 0
    attempts = 0

    while valid < PAIRS_PER_SCENE and attempts < 200:
        attempts += 1
        idx1 = np.random.randint(0, max(1, n_imgs - FRAME_GAP - 1))
        idx2 = idx1 + FRAME_GAP
        if idx2 >= n_imgs:
            continue

        x = np.random.randint(50, 590)
        y = np.random.randint(50, 430)

        ok = plot_quad(all_imgs, K_intr, data_dir, scene_name,
                       idx1, idx2, {f'Point_{total_plotted}': (x, y)})
        if ok:
            valid += 1
            total_plotted += 1

    print(f'  Plotted {valid}/{PAIRS_PER_SCENE} for {scene_name}')

print(f'\nDone. Total plots: {total_plotted}')